# 02 - Library Baseline: better_profanity

Evaluate the `better_profanity` library as a rule-based baseline for Bangla toxic comment detection.

In [ ]:
import pandas as pd
import time
from better_profanity import profanity
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import sys
sys.path.append('..')
from src.utils import load_config, save_metrics, ensure_dirs

config = load_config('../configs/config.yaml')
ensure_dirs('../results', '../plots')

## Load Test Data

In [ ]:
DATA_PATH = '../data/test.csv'
TEXT_COL = config['data']['text_column']
LABEL_COL = config['data']['label_column']

df = pd.read_csv(DATA_PATH)
texts = df[TEXT_COL].astype(str).tolist()
labels = df[LABEL_COL].astype(int).tolist()
print(f'Loaded {len(texts)} test samples')

## Run better_profanity

This library is English-focused, so it serves as a lower-bound baseline showing why Bangla-specific models are needed.

In [ ]:
profanity.load_censor_words()

start_time = time.time()
predictions = [1 if profanity.contains_profanity(text) else 0 for text in texts]
inference_time = time.time() - start_time

print(f'Inference time: {inference_time:.2f}s')
print(f'Samples/second: {len(texts) / inference_time:.0f}')
print(f'Predicted toxic: {sum(predictions)} / {len(predictions)}')

## Evaluation

In [ ]:
metrics = {
    'model': 'better_profanity',
    'accuracy': accuracy_score(labels, predictions),
    'f1': f1_score(labels, predictions, average='weighted'),
    'precision': precision_score(labels, predictions, average='weighted', zero_division=0),
    'recall': recall_score(labels, predictions, average='weighted'),
    'inference_time': inference_time,
    'samples_per_second': len(texts) / inference_time,
}

print('\n=== better_profanity Baseline Results ===')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\n=== Classification Report ===')
print(classification_report(labels, predictions, target_names=['non-toxic', 'toxic'], zero_division=0))

print('\n=== Confusion Matrix ===')
print(confusion_matrix(labels, predictions))

In [ ]:
save_metrics(metrics, '../results/baseline_better_profanity.json')
print('Results saved.')